In [ ]:
import os
import json
import pathlib
import asyncio
from tqdm.notebook import tqdm
import logging
import time
from typing import List
from datetime import datetime


from helpers.constants import OPENALEX_NL_UAS


pdf_dir = pathlib.Path("./outputs/downloaded_pdfs")
markdown_output_dir = pathlib.Path("./outputs/markdown_output")
json_output_dir = pathlib.Path("./outputs/json_output")
log_output_dir = pathlib.Path("./outputs/logs")
filtered_grant_output_dir = pathlib.Path("./outputs/json_filtered_grant_outputs")

for d in (pdf_dir, markdown_output_dir, json_output_dir, log_output_dir):
    d.mkdir(parents=True, exist_ok=True)

context_path = pathlib.Path("./helpers/context.md")
with context_path.open("r", encoding="utf-8") as f:
    system_prompt_text = f.read()

with open("./helpers/Schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)


In [ ]:
# Check JSON outputs
import json
from collections import defaultdict

print_files = False # print matching file names?
print_counts = False # print project name counts?

count_project_name = 0
count_grant_number = 0
count_both = 0

matched_project_name = []
matched_grant_number = []
matched_both = []

project_name_counts = defaultdict(int)

for json_path in sorted(json_output_dir.glob("*.json")):
    try:
        with json_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        agent_data = data.get("agent", {})
        project_name = agent_data.get("projectName")
        grant = agent_data.get("grantNumber")
        
        has_project = project_name is not None
        has_grant = grant is not None
        
        if has_project:
            count_project_name += 1
            matched_project_name.append(json_path.name)
            project_name_counts[project_name] += 1
        if has_grant:
            count_grant_number += 1
            matched_grant_number.append(json_path.name)
        if has_project and has_grant:
            count_both += 1
            matched_both.append(json_path.name)
    except Exception as e:
        print(f"[!] Skipped {json_path.name}: {e}")

print(f"Files with projectName: {count_project_name}")
if print_files:
    for name in matched_project_name:
        print(f"  {name}")

print(f"Files with grantNumber: {count_grant_number}")
if print_files:
    for name in matched_grant_number:
        print(f"  {name}")

print(f"Files with both: {count_both}")
if print_files:
    for name in matched_both:
        print(f"  {name}")

if print_counts:
    print("Project name counts:")
    for name, count in sorted(project_name_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {name}: {count}")

In [ ]:
# Count and list files with grant number starting with NWA or NWO, or funding agency containing NWO or Nederlandse Organisatie voor Wetenschappelijk Onderzoek
import json


print_files = True
nwo_related_count = 0
nwo_related_files = []

print(f"Checking for grants starting with NWA or NWO, or funding agencies containing NWO or Nederlandse Organisatie voor Wetenschappelijk Onderzoek in {filtered_grant_output_dir}...")

for json_path in sorted(filtered_grant_output_dir.glob("*.json")):
    try:
        with json_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
            
        agent_data = data.get("agent", {})
        grant = agent_data.get("grantNumberNWOHeuristic")
        funding_agency = agent_data.get("fundingOrgAgency")
        
        matches = False
        
        if grant and isinstance(grant, str):
            grant_upper = grant.upper().strip()
            if grant_upper.startswith("NWA") or grant_upper.startswith("NWO"):
                matches = True
                
        if funding_agency and isinstance(funding_agency, str):
            agency_upper = funding_agency.upper()
            if "NWO" in agency_upper or "NEDERLANDSE ORGANISATIE VOOR WETENSCHAPPELIJK ONDERZOEK" in agency_upper:
                matches = True
                
        if matches:
            nwo_related_count += 1
            nwo_related_files.append((json_path.name, grant, funding_agency))
                
    except Exception as e:
        print(f"Error reading {json_path.name}: {e}")

print(f"\nFound {nwo_related_count} files with matching grants or funding agencies.")

if print_files and nwo_related_files:
    print("\nFiles found:")
    for fname, gname, agency in nwo_related_files:
        grant_info = f"Grant: {gname}" if gname else ""
        agency_info = f"Agency: {agency}" if agency else ""
        info = " | ".join(filter(None, [grant_info, agency_info]))
        print(f"  {fname} ({info})")